<a href="https://colab.research.google.com/github/amankiitg/LLM_Prod/blob/main/Memory_Testing_Dashboard_with_Qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Memory Testing Dashboard with Qwen

This teaching-guide style Colab notebook helps you **explore, test, and visualize LLM memory behaviors** using Qwen and multiple memory architectures (Buffer, Summary, Episodic, and Hybrid).

**What you'll do here:**
- Install required libraries
- Initialize the LLM (Qwen2.5-1.5B-Instruct on GPU; DialoGPT fallback on CPU)
- Build a small **knowledge base** from a demo PDF (Vaswani et al., *Attention Is All You Need*)
- Plug in **advanced memory systems** (buffer, summary, episodic retrieval, hybrid)
- Launch a **Gradio dashboard** to interactively test memory recall & retrieval



## Learning Objectives

By the end of this notebook, you will be able to:
1. **Explain** how different memory systems work for chat LLMs.
2. **Run** a minimal **RAG-style** pipeline leveraging embeddings + FAISS.
3. **Compare** the behavior of Buffer, Summary, and Episodic memories.
4. **Operate** an interactive **Gradio** dashboard to test memory recall.

---

## Quick Start
1. Run **Cell 1** to install dependencies.
2. Run **Cell 2–7** to load models, set up memory systems, and prepare the knowledge base.
3. Run **Cells 8–10** to start the **Gradio dashboard** and experiment:
   - Choose **Episodic Memory** → **Start Chat**
   - Ask: *"What is attention in transformers?"*
   - Ask: *"How does self-attention work?"*
   - Ask: *"What did we discuss about attention?"*
   - Watch the **Memory State** and **Retrieved Context** panels update!


## Cell 1 — Install Dependencies
Install all required libraries. If you restart your runtime, re-run this cell.

In [1]:

# ================================
# CELL 1: Install Dependencies
# ================================
!pip install torch transformers sentence-transformers faiss-cpu pypdf gradio requests numpy accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 29.6 MB/s eta 0:00:00



## Cell 2 — Imports & Runtime Setup
This cell:
- Imports Python libraries
- Detects GPU
- Configures model names (Qwen on GPU, DialoGPT on CPU fallback)
- Sets chunk sizes and paths


In [2]:

# ================================
# CELL 2: Imports and Setup
# ================================
import os, json, time, uuid, shutil, textwrap, math, random, string, pathlib, gc
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss
from pypdf import PdfReader
import gradio as gr
import requests

# Device selection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

# Configuration - Using Qwen
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
# Choose model based on available resources
if torch.cuda.is_available():
    GEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Good balance of quality and speed
    print("🚀 Using Qwen2.5-1.5B-Instruct (GPU)")
else:
    GEN_MODEL_NAME = "microsoft/DialoGPT-medium"  # Fallback for CPU
    print("⚡ Using DialoGPT-medium (CPU fallback)")

CHUNK_SIZE = 900
CHUNK_OVERLAP = 250
DOC_TOP_K = 3  # Reduced for better context management
BUFFER_TURNS = 6
EPISODIC_TOP_K = 5
SUMMARY_UPDATE_EVERY = 3

# Paths
DATA_DIR = "/content/data"
PDF_DIR = os.path.join(DATA_DIR, "pdfs")
SESSION_ID = str(uuid.uuid4())[:8]

os.makedirs(PDF_DIR, exist_ok=True)
print(f"Session ID: {SESSION_ID}")


Using device: cuda
GPU available: True
GPU name: Tesla T4
🚀 Using Qwen2.5-1.5B-Instruct (GPU)
Session ID: b88177f1



## Cell 3 — Memory Classes
We define **BufferMemory**, **SummaryMemory**, **EpisodicMemory**, and a **HybridMemory** that combines all three.
- *Buffer:* recent turns
- *Summary:* rolling conversation summary (auto-updates every N turns)
- *Episodic:* FAISS-backed semantic retrieval of past episodes
- *Hybrid:* best of all worlds


In [3]:

# ================================
# CELL 3: Memory Classes
# ================================

@dataclass
class ChatTurn:
    user: str
    assistant: str
    ts: float

class BaseMemory:
    def get_context(self, query: str) -> str:
        return ""
    def add_turn(self, user: str, assistant: str):
        pass
    def save(self, path: str):
        pass
    def load(self, path: str):
        pass

class NullMemory(BaseMemory):
    name = "no_memory"

class BufferMemory(BaseMemory):
    name = "short_term_buffer"
    def __init__(self, max_turns: int = 6):
        self.max_turns = max_turns
        self.history: List[ChatTurn] = []

    def get_context(self, query: str) -> str:
        if not self.history:
            return ""

        turns = self.history[-self.max_turns:]
        ctx_lines = []
        for i, t in enumerate(turns, 1):
            ctx_lines.append(f"Previous exchange {i}:")
            ctx_lines.append(f"User: {t.user}")
            ctx_lines.append(f"Assistant: {t.assistant}")
            ctx_lines.append("")
        return "\n".join(ctx_lines).strip()

    def add_turn(self, user: str, assistant: str):
        self.history.append(ChatTurn(user=user, assistant=assistant, ts=time.time()))
        if len(self.history) > 1000:
            self.history = self.history[-1000:]

class SummaryMemory(BaseMemory):
    name = "long_term_summary"
    def __init__(self, llm, update_every: int = 3):
        self.llm = llm
        self.update_every = update_every
        self.history: List[ChatTurn] = []
        self.summary: str = ""

    def _should_update(self) -> bool:
        return len(self.history) > 0 and (len(self.history) % self.update_every == 0)

    def _format_history(self) -> str:
        lines = []
        for t in self.history[-6:]:  # Last 6 turns for summary
            lines.append(f"User: {t.user}")
            lines.append(f"Assistant: {t.assistant}")
        return "\n".join(lines)

    def get_context(self, query: str) -> str:
        if not self.summary.strip():
            return ""
        return f"Conversation summary so far:\n{self.summary}"

    def add_turn(self, user: str, assistant: str):
        self.history.append(ChatTurn(user=user, assistant=assistant, ts=time.time()))

        if self._should_update():
            recent_text = self._format_history()

            if self.summary.strip():
                prompt = f"""Update this conversation summary with the new information:

Current Summary:
{self.summary}

Recent Conversation:
{recent_text}

Updated Summary (keep it concise, focus on key topics discussed):"""
            else:
                prompt = f"""Create a concise summary of this conversation focusing on the main topics and key information discussed:

{recent_text}

Summary:"""

            try:
                self.summary = self.llm.generate(prompt, max_new_tokens=150)
            except Exception as e:
                print(f"Summary update failed: {e}")

class EpisodicMemory(BaseMemory):
    name = "episodic_semantic"
    def __init__(self, embedder: SentenceTransformer, dim: int):
        self.embedder = embedder
        self.dim = dim
        self.index = faiss.IndexFlatIP(dim)
        self.episodes: List[Dict[str, Any]] = []
        self._embs = None

    def get_context(self, query: str) -> str:
        if len(self.episodes) == 0:
            return ""

        try:
            q = self.embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
            D, I = self.index.search(q, min(EPISODIC_TOP_K, len(self.episodes)))

            lines = []
            for idx, score in zip(I[0], D[0]):
                if idx == -1 or score < 0.15:  # Skip low relevance
                    continue
                episode = self.episodes[int(idx)]
                lines.append(f"Relevant past conversation (similarity: {score:.2f}):\n{episode['text']}")

            return "\n\n".join(lines)
        except Exception as e:
            print(f"Episodic retrieval error: {e}")
            return ""

    def add_turn(self, user: str, assistant: str):
        episode_text = f"User: {user}\nAssistant: {assistant}"
        self.episodes.append({"text": episode_text, "ts": time.time()})

        try:
            emb = self.embedder.encode([episode_text], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
            if self._embs is None:
                self._embs = emb
                self.index = faiss.IndexFlatIP(self.dim)
                self.index.add(emb)
            else:
                self._embs = np.vstack([self._embs, emb])
                self.index.add(emb)
        except Exception as e:
            print(f"Episode embedding error: {e}")

class HybridMemory(BaseMemory):
    name = "hybrid"
    def __init__(self, buffer: BufferMemory, summary: SummaryMemory, episodic: EpisodicMemory):
        self.buffer = buffer
        self.summary = summary
        self.episodic = episodic

    def get_context(self, query: str) -> str:
        parts = []

        # 1) Episodic first (so it survives any truncation)
        e = self.episodic.get_context(query)
        if e.strip():
            parts.append("RELEVANT PAST DISCUSSIONS:\n" + e)

        # 2) Then summary (it can be long)
        s = self.summary.get_context(query)
        if s.strip():
            parts.append("CONVERSATION SUMMARY:\n" + s)

        # 3) Then recent buffer if we still have room to add context
        b = self.buffer.get_context(query)
        if b.strip() and len(parts) < 2:
            parts.append("RECENT CONVERSATION:\n" + b)

        return "\n\n".join(parts)


    def add_turn(self, user: str, assistant: str):
        self.buffer.add_turn(user, assistant)
        self.summary.add_turn(user, assistant)
        self.episodic.add_turn(user, assistant)



## Cell 4 — Qwen LLM Wrapper
A minimal LLM utility to:
- Load **Qwen** (or issue a CPU fallback)
- Generate responses with a chat template when applicable
- Provide a helper for summaries


In [4]:

# ================================
# CELL 4: Qwen LLM Class
# ================================

class QwenLLM:
    def __init__(self, model_name: str = GEN_MODEL_NAME, device: str = DEVICE):
        self.model_name = model_name
        self.device = device

        print(f"Loading {model_name}...")

        try:
            # Load tokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                trust_remote_code=True,
                padding_side="left"
            )

            # Add pad token if missing
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            # Load model with appropriate settings
            if "Qwen" in model_name:
                model_kwargs = {
                    "trust_remote_code": True,
                    "torch_dtype": torch.float16 if device == "cuda" else torch.float32,
                    "device_map": "auto" if device == "cuda" else None
                }
            else:
                # For DialoGPT fallback
                model_kwargs = {
                    "torch_dtype": torch.float32
                }

            self.model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

            if device == "cpu":
                self.model = self.model.to(device)

            print(f" Model loaded successfully on {device}")

        except Exception as e:
            print(f" Failed to load {model_name}: {e}")
            print(" Falling back to DialoGPT...")
            # Fallback to a simpler model
            self.model_name = "microsoft/DialoGPT-small"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.model = AutoModelForCausalLM.from_pretrained(self.model_name).to(device)

    def generate(self, prompt: str, max_new_tokens: int = 200) -> str:
        try:
            # Format prompt for chat models
            if "Qwen" in self.model_name:
                messages = [{"role": "user", "content": prompt}]
                formatted_prompt = self.tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
            else:
                formatted_prompt = prompt

            # Tokenize with length limit
            inputs = self.tokenizer(
                formatted_prompt,
                return_tensors="pt",
                truncation=True,
                max_length=2048
            ).to(self.device)

            # Generate
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    repetition_penalty=1.1
                )

            # Decode response (only new tokens)
            response = self.tokenizer.decode(
                outputs[0][inputs.input_ids.shape[-1]:],
                skip_special_tokens=True
            ).strip()

            return response if response else "I apologize, I couldn't generate a proper response."

        except Exception as e:
            print(f"Generation error: {e}")
            return f"I encountered an error: {str(e)}"

    def summarize(self, text: str, max_new_tokens: int = 150) -> str:
        prompt = f"""Please create a concise summary of the following conversation, focusing on the main topics and key points discussed:

{text}

Summary:"""
        return self.generate(prompt, max_new_tokens)



## Cell 5 — Supporting Utilities
- Download a demo PDF (Vaswani et al. 2017)
- Read PDFs and chunk into passages
- Build a **FAISS** index with **Sentence-Transformers** embeddings


In [5]:

# ================================
# CELL 5: Supporting Classes
# ================================

def download_demo_pdfs():
    """Download demo PDFs for testing"""
    demo_links = [
        "https://arxiv.org/pdf/1706.03762.pdf",  # Attention Is All You Need
    ]
    os.makedirs(PDF_DIR, exist_ok=True)
    saved = []

    print("📚 Downloading demo PDF...")
    for url in demo_links:
        filename = os.path.join(PDF_DIR, os.path.basename(url))
        if not os.path.exists(filename):
            print(f"  Downloading {os.path.basename(url)}...")
            try:
                r = requests.get(url, timeout=60)
                with open(filename, "wb") as f:
                    f.write(r.content)
                print(f"   Downloaded {os.path.basename(url)}")
            except Exception as e:
                print(f"   Failed to download: {e}")
                continue
        else:
            print(f"  ⏭️ {os.path.basename(url)} already exists")
        saved.append(filename)

    return saved

def load_pdfs_to_docs(pdf_paths: List[str]) -> List[Dict[str, Any]]:
    docs = []
    for path in pdf_paths:
        try:
            reader = PdfReader(path)
            for i, page in enumerate(reader.pages):
                try:
                    text = page.extract_text() or ""
                except Exception:
                    text = ""
                text = text.replace("\x00", " ").strip()
                if text and len(text) > 50:  # Skip very short pages
                    docs.append({
                        "text": text,
                        "source": os.path.basename(path),
                        "page": i + 1,
                    })
        except Exception as e:
            print(f"[WARN] Could not read {path}: {e}")
    return docs

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        chunks.append(chunk)
        if end == len(text):
            break
        start = end - overlap
    return chunks

def chunk_docs(docs: List[Dict[str, Any]], chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    out = []
    for d in docs:
        for ch in chunk_text(d["text"], chunk_size, overlap):
            if len(ch.strip()) > 100:  # Skip very short chunks
                out.append({
                    "text": ch,
                    "source": d["source"],
                    "page": d["page"]
                })
    return out

class KnowledgeBase:
    def __init__(self, embed_model_name: str = EMBED_MODEL_NAME):
        print(f"Loading embedder {embed_model_name}...")
        self.embedder = SentenceTransformer(embed_model_name)
        self.dim = self.embedder.get_sentence_embedding_dimension()
        self.chunks = []
        self.index = None
        self.built = False

    def build_from_files(self, file_paths: List[str], chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
        docs = load_pdfs_to_docs(file_paths)
        if len(docs) == 0:
            raise ValueError("No extractable text found in PDFs.")

        print(f"📄 Loaded {len(docs)} pages from PDFs")
        chunks = chunk_docs(docs, chunk_size=chunk_size, overlap=overlap)
        print(f"✂️ Created {len(chunks)} chunks")

        texts = [c["text"] for c in chunks]

        print("Creating embeddings...")
        embs = self.embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)

        self.index = faiss.IndexFlatIP(self.dim)
        self.index.add(embs.astype(np.float32))
        self.chunks = chunks
        self.built = True

        print(f"Knowledge base built with {len(chunks)} chunks")
        return len(chunks)

    def retrieve(self, query: str, top_k: int = DOC_TOP_K) -> List[Dict[str, Any]]:
        if not self.built:
            return []

        try:
            q = self.embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
            D, I = self.index.search(q.astype(np.float32), min(top_k, len(self.chunks)))

            results = []
            for idx, score in zip(I[0], D[0]):
                if idx == -1 or score < 0.1:  # Skip very low relevance
                    continue
                chunk = dict(self.chunks[idx])
                chunk["_score"] = float(score)
                results.append(chunk)
            return results
        except Exception as e:
            print(f"Retrieval error: {e}")
            return []



## Cell 6 — Enhanced Chat Engine
The chat engine wires together the **KnowledgeBase**, **LLM**, and a selected **Memory** to prepare prompts and generate helpful answers.


In [6]:

# ================================
# CELL 6: Enhanced Chat Engine
# ================================

class ChatEngine:
    def __init__(self, kb, llm, memory):
        self.kb = kb
        self.llm = llm
        self.memory = memory

    def answer(self, user_input: str) -> Tuple[str, List[Dict[str, Any]], str]:
        # Get relevant documents
        retrieved = self.kb.retrieve(user_input, top_k=DOC_TOP_K) if self.kb and self.kb.built else []

        # Get memory context
        memory_context = self.memory.get_context(user_input) if self.memory else ""

        # Limit memory context length
        if len(memory_context) > 800:
            memory_context = memory_context[:800] + "\n... (memory truncated for clarity)"

        # Build document context
        doc_context = ""
        if retrieved:
            doc_parts = []
            for i, r in enumerate(retrieved, 1):
                snippet = r['text'][:400] + "..." if len(r['text']) > 400 else r['text']
                doc_parts.append(f"[Source {i}] {r['source']} page {r['page']}:\n{snippet}")
            doc_context = "\n\n".join(doc_parts)

        # Build comprehensive prompt
        prompt_parts = []
        prompt_parts.append("You are a helpful AI assistant. Answer the user's question based on the provided context and conversation memory.")

        if memory_context:
            prompt_parts.append(f"\nCONVERSATION CONTEXT:\n{memory_context}")

        if doc_context:
            prompt_parts.append(f"\nRELEVANT DOCUMENTS:\n{doc_context}")

        prompt_parts.append(f"\nUSER QUESTION: {user_input}")
        prompt_parts.append("\nPlease provide a helpful and accurate answer based on the above information:")

        prompt = "\n".join(prompt_parts)

        # Generate response
        try:
            answer = self.llm.generate(prompt, max_new_tokens=250)

            # Clean up response
            answer = answer.strip()
            if not answer or len(answer) < 10:
                answer = "I apologize, but I'm having trouble generating a complete response. Could you please rephrase your question?"

        except Exception as e:
            answer = f"I encountered an error while processing your question: {str(e)}"

        return answer, retrieved, memory_context

    def record_turn(self, user: str, assistant: str):
        if self.memory:
            self.memory.add_turn(user, assistant)



## Cell 7 — Initialize System
- Download the demo PDF and build the **KnowledgeBase**
- Load the **LLM**
- Initialize all **Memory** types (and a **Hybrid**)


In [7]:

# ================================
# CELL 7: Initialize System
# ================================

print("Initializing Enhanced Memory Testing Dashboard...")

# Download and setup knowledge base
print("Setting up knowledge base...")
pdf_files = download_demo_pdfs()
if pdf_files:
    kb = KnowledgeBase(EMBED_MODEL_NAME)
    n_chunks = kb.build_from_files(pdf_files)
    print(f"Knowledge base ready with {n_chunks} chunks")
else:
    print("No PDFs available - some features may be limited")
    kb = None

# Initialize LLM
print(" Loading language model...")
llm = QwenLLM(GEN_MODEL_NAME, DEVICE)
print(" Language model ready")

# Initialize all memory types
print("Setting up memory systems...")
buffer_mem = BufferMemory(max_turns=BUFFER_TURNS)
summary_mem = SummaryMemory(llm=llm, update_every=SUMMARY_UPDATE_EVERY)

if kb:
    episodic_mem = EpisodicMemory(embedder=kb.embedder, dim=kb.dim)
    hybrid_mem = HybridMemory(buffer=buffer_mem, summary=summary_mem, episodic=episodic_mem)
else:
    # Fallback if no knowledge base
    temp_embedder = SentenceTransformer(EMBED_MODEL_NAME)
    episodic_mem = EpisodicMemory(embedder=temp_embedder, dim=temp_embedder.get_sentence_embedding_dimension())
    hybrid_mem = HybridMemory(buffer=buffer_mem, summary=summary_mem, episodic=episodic_mem)

memories = {
    "None (No Memory)": NullMemory(),
    "Buffer Memory": buffer_mem,
    "Summary Memory": summary_mem,
    "Episodic Memory": episodic_mem,
    "Hybrid Memory": hybrid_mem
}

print("All systems initialized and ready!")


Initializing Enhanced Memory Testing Dashboard...
Setting up knowledge base...
📚 Downloading demo PDF...
   Downloaded 1706.03762.pdf
Loading embedder sentence-transformers/all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

 Model loaded successfully on cuda
 Language model ready
Setting up memory systems...
All systems initialized and ready!



## Cell 8 — Dashboard Functions
Helper functions for:
- Starting a chat session
- Processing a chat turn
- Running test scenarios
- Clearing memory
- Formatting memory state


In [8]:

# ================================
# CELL 8: Dashboard Functions
# ================================

# Global state
current_engine = None
conversation_history = []

def format_memory_state(memory, memory_name):
    """Format current memory state for display"""
    if memory_name == "None (No Memory)":
        return " No memory - each conversation is independent"

    elif memory_name == "Buffer Memory":
        if hasattr(memory, 'history') and memory.history:
            recent = memory.history[-3:]  # Show last 3
            lines = [f" Buffer Memory ({len(memory.history)}/{memory.max_turns} turns)\n"]
            lines.append("Recent conversations:")
            for i, turn in enumerate(recent):
                lines.append(f"  {len(memory.history)-len(recent)+i+1}. User: {turn.user[:60]}...")
                lines.append(f"      Bot: {turn.assistant[:60]}...")
            return "\n".join(lines)
        return f" Buffer Memory: Empty (0/{getattr(memory, 'max_turns', 6)} turns)"

    elif memory_name == "Summary Memory":
        if hasattr(memory, 'summary') and memory.summary:
            summary_preview = memory.summary[:300] + "..." if len(memory.summary) > 300 else memory.summary
            return f"Summary Memory ({len(memory.summary)} chars)\n\nSummary:\n{summary_preview}\n\nTurns processed: {len(getattr(memory, 'history', []))}"
        return f"Summary Memory: No summary yet\nProcessed {len(getattr(memory, 'history', []))} turns"

    elif memory_name == "Episodic Memory":
        if hasattr(memory, 'episodes') and memory.episodes:
            recent = memory.episodes[-2:]
            lines = [f"Episodic Memory ({len(memory.episodes)} episodes)\n"]
            lines.append("Recent episodes:")
            for i, episode in enumerate(recent):
                episode_preview = episode['text'][:100].replace('\n', ' → ') + "..."
                lines.append(f"  {len(memory.episodes)-len(recent)+i+1}. {episode_preview}")
            return "\n".join(lines)
        return "Episodic Memory: No episodes stored yet"

    elif memory_name == "Hybrid Memory":
        lines = [f"Hybrid Memory - Combined System\n"]
        if hasattr(memory, 'buffer') and memory.buffer.history:
            lines.append(f"   Buffer: {len(memory.buffer.history)} recent turns")
        if hasattr(memory, 'summary') and memory.summary.summary:
            lines.append(f"   Summary: {len(memory.summary.summary)} characters")
        if hasattr(memory, 'episodic') and memory.episodic.episodes:
            lines.append(f"   Episodes: {len(memory.episodic.episodes)} stored")

        if len(lines) == 1:
            lines.append("  All components empty")

        return "\n".join(lines)

    return " Unknown memory type"

def get_memory_retrieved_context(memory_context, memory_name, query):
    """Format retrieved memory context for display (no truncation)."""
    if memory_name == "None (No Memory)":
        return " No context retrieved (no memory system)"
    if not memory_context.strip():
        return f" No relevant context found in {memory_name}"
    return f" Retrieved from {memory_name}:\n\n{memory_context}"


def start_chat(memory_choice):
    """Initialize chat with selected memory type"""
    global current_engine, conversation_history

    memory = memories[memory_choice]
    current_engine = ChatEngine(kb=kb, llm=llm, memory=memory)
    conversation_history = []

    memory_state = format_memory_state(memory, memory_choice)

    return (
        f" Chat started with **{memory_choice}**\n\nReady to test memory behavior with Qwen!",
        memory_state,
        " Ask a question to see how memory retrieval works",
        []
    )

def chat_turn(message, memory_choice, chat_history):
    """Process a single chat turn"""
    global current_engine, conversation_history

    if not current_engine or not message.strip():
        return chat_history, " Please start chat first or enter a message", "No context"

    try:
        # Generate answer and get contexts
        answer, doc_sources, memory_context = current_engine.answer(message)
        full_memory_context = current_engine.memory.get_context(message) if current_engine.memory else ""
        current_engine.record_turn(message, answer)

        # Format answer with sources
        answer_with_sources = answer
        if doc_sources:
            source_info = "\n\n **Sources:** " + ", ".join([
                f"{s['source']} p.{s['page']} (score: {s['_score']:.2f})"
                for s in doc_sources[:3]
            ])
            answer_with_sources += source_info
        else:
            answer_with_sources += "\n\n📭 *(No document sources found)*"

        # Update displays
        chat_history = chat_history + [[message, answer_with_sources]]
        memory_state = format_memory_state(memories[memory_choice], memory_choice)
        retrieved_context = get_memory_retrieved_context(full_memory_context, memory_choice, message)


        return chat_history, memory_state, retrieved_context

    except Exception as e:
        error_msg = f" Error: {str(e)}"
        return chat_history + [[message, error_msg]], "Error occurred", "Error in retrieval"

def run_test_scenario(scenario_name, memory_choice, chat_history):
    """Run predefined test scenarios"""
    global current_engine

    scenarios = {
        "Attention Mechanism Test": [
            "What is the attention mechanism in transformers?",
            "How does self-attention work?",
            "What did we discuss about attention earlier?"
        ],
        "Context Switch Test": [
            "What is a transformer?",
            "Tell me about neural networks",
            "Going back to transformers, what makes them special?"
        ],
        "Memory Recall Test": [
            "Explain positional encoding",
            "What about layer normalization?",
            "Can you summarize what we've discussed so far?"
        ]
    }

    if scenario_name not in scenarios or not current_engine:
        return chat_history, " Please start chat first", "No scenario run"

    questions = scenarios[scenario_name]
    memory = memories[memory_choice]

    for i, question in enumerate(questions, 1):
        try:
            answer, doc_sources, memory_context = current_engine.answer(question)
            current_engine.record_turn(question, answer)

            # Format answer with scenario info
            scenario_prefix = f"**[Scenario {i}/{len(questions)}]** "
            if doc_sources:
                source_info = f" 📚 [{len(doc_sources)} sources]"
                answer_formatted = scenario_prefix + answer + source_info
            else:
                answer_formatted = scenario_prefix + answer

            chat_history = chat_history + [[question, answer_formatted]]

        except Exception as e:
            chat_history = chat_history + [[question, f" Error in scenario: {str(e)}"]]

    # Final update
    memory_state = format_memory_state(memory, memory_choice)
    retrieved_context = f" Completed '{scenario_name}' scenario with {len(questions)} questions"

    return chat_history, memory_state, retrieved_context

def clear_memory(memory_choice):
    """Clear the selected memory type"""
    global current_engine, conversation_history

    # Re-initialize the specific memory type
    if memory_choice == "Buffer Memory":
        memories[memory_choice] = BufferMemory(max_turns=BUFFER_TURNS)
    elif memory_choice == "Summary Memory":
        memories[memory_choice] = SummaryMemory(llm=llm, update_every=SUMMARY_UPDATE_EVERY)
    elif memory_choice == "Episodic Memory":
        if kb:
            memories[memory_choice] = EpisodicMemory(embedder=kb.embedder, dim=kb.dim)
        else:
            temp_embedder = SentenceTransformer(EMBED_MODEL_NAME)
            memories[memory_choice] = EpisodicMemory(embedder=temp_embedder, dim=temp_embedder.get_sentence_embedding_dimension())
    elif memory_choice == "Hybrid Memory":
        buffer_mem = BufferMemory(max_turns=BUFFER_TURNS)
        summary_mem = SummaryMemory(llm=llm, update_every=SUMMARY_UPDATE_EVERY)
        if kb:
            episodic_mem = EpisodicMemory(embedder=kb.embedder, dim=kb.dim)
        else:
            temp_embedder = SentenceTransformer(EMBED_MODEL_NAME)
            episodic_mem = EpisodicMemory(embedder=temp_embedder, dim=temp_embedder.get_sentence_embedding_dimension())
        memories[memory_choice] = HybridMemory(buffer=buffer_mem, summary=summary_mem, episodic=episodic_mem)

    # Update engine if it exists
    if current_engine:
        current_engine.memory = memories[memory_choice]

    conversation_history = []
    memory_state = format_memory_state(memories[memory_choice], memory_choice)

    return [], memory_state, "Memory cleared", f"✅ {memory_choice} has been reset"



## Cell 9 — Create Gradio Interface
This builds the interactive UI:
- Pick a memory type
- Start chat, send messages
- Run predefined scenarios
- Inspect memory state and retrieved context


In [9]:

# ================================
# CELL 9: Create Gradio Interface
# ================================

def create_dashboard():
    with gr.Blocks(title=" Memory Testing Dashboard with Qwen", theme=gr.themes.Soft()) as dashboard:

        gr.HTML("""
<link rel='preconnect' href='https://fonts.googleapis.com'>
<link rel='preconnect' href='https://fonts.gstatic.com' crossorigin>
<link href='https://fonts.googleapis.com/css2?family=Figtree:wght@400;600;700&display=swap' rel='stylesheet'>

<style>
  .figtree * {
    font-family: 'Figtree', system-ui, -apple-system, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif;
    color: #ffffff !important;
  }
</style>

<div class='figtree' style='text-align: center; padding: 20px; background: linear-gradient(90deg, #667eea 0%, #764ba2 100%); border-radius: 10px; margin-bottom: 20px;'>
  <h1 style='margin: 0; font-size: 2.5rem;'>Memory Testing Dashboard</h1>
  <p style='margin: 10px 0 0 0; font-size: 1.2rem; opacity: 0.9;'>Interactive testing with Qwen LLM and advanced memory architectures</p>
</div>
""")


        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 🎛️ Controls")

                memory_dropdown = gr.Dropdown(
                    choices=list(memories.keys()),
                    value="Hybrid Memory",
                    label="Memory Type",
                    info="Select which memory system to test"
                )

                start_btn = gr.Button("🚀 Start Chat", variant="primary", size="lg")

                with gr.Row():
                    clear_btn = gr.Button("🗑️ Clear Memory", variant="secondary")

                status_display = gr.Markdown("Select a memory type and click 'Start Chat'")

                gr.Markdown("### 🧪 Test Scenarios")
                scenario_dropdown = gr.Dropdown(
                    choices=["Attention Mechanism Test", "Context Switch Test", "Memory Recall Test"],
                    label="Quick Test Scenarios",
                    info="Run predefined test sequences"
                )
                scenario_btn = gr.Button("▶️ Run Scenario", variant="secondary")

            with gr.Column(scale=2):
                gr.Markdown("### 💬 Conversation")

                chatbot = gr.Chatbot(
                    height=400,
                    label="Chat History",
                    show_label=True,
                    avatar_images=("👨‍💻", "🤖")
                )

                with gr.Row():
                    msg_input = gr.Textbox(
                        placeholder="Ask about transformers, attention, or anything else...",
                        label="Your Message",
                        scale=4
                    )
                    send_btn = gr.Button("📨 Send", variant="primary", scale=1)

        with gr.Row():
            with gr.Column():
                gr.Markdown("### 🧠 Memory State")
                memory_state_display = gr.Textbox(
                    label="Current Memory Contents",
                    lines=8,
                    max_lines=8,
                    value="No memory initialized yet"
                )

            with gr.Column():
                gr.Markdown("### 🔍 Retrieved Context")
                context_display = gr.Textbox(
                    label="What Memory Retrieved for Last Query",
                    lines=20,
                    value="No query processed yet"
                )

        gr.Markdown("""
        ### 📋 How to Test Memory Types:

        **Quick Start:**
        1. Select a memory type and click "Start Chat"
        2. Ask questions about transformers/attention
        3. Watch how each memory type behaves differently

        **Test Scenarios:**
        - **Attention Test**: Ask about attention, then reference previous discussion
        - **Context Switch**: Change topics, then return to previous topic
        - **Memory Recall**: Test ability to summarize conversation

        **Memory Types:**
        - **None**: No memory between turns
        - **Buffer**: Remembers last 6 turns
        - **Summary**: Creates running summary every 3 turns
        - **Episodic**: Semantic search through all past conversations
        - **Hybrid**: Combines all three memory types

        **Best Test Sequence:**
        1. Start with "Episodic Memory"
        2. Ask: "What is attention in transformers?"
        3. Ask: "How does self-attention work?"
        4. Ask: "What did we discuss about attention?"
        5. Watch the retrieval panels to see memory in action!
        """)

        # Event handlers
        start_btn.click(
            start_chat,
            inputs=[memory_dropdown],
            outputs=[status_display, memory_state_display, context_display, chatbot]
        )

        def send_message(message, memory_choice, chat_history):
            return chat_turn(message, memory_choice, chat_history)

        send_btn.click(
            send_message,
            inputs=[msg_input, memory_dropdown, chatbot],
            outputs=[chatbot, memory_state_display, context_display]
        ).then(
            lambda: "",
            outputs=[msg_input]
        )

        msg_input.submit(
            send_message,
            inputs=[msg_input, memory_dropdown, chatbot],
            outputs=[chatbot, memory_state_display, context_display]
        ).then(
            lambda: "",
            outputs=[msg_input]
        )

        scenario_btn.click(
            run_test_scenario,
            inputs=[scenario_dropdown, memory_dropdown, chatbot],
            outputs=[chatbot, memory_state_display, context_display]
        )

        clear_btn.click(
            clear_memory,
            inputs=[memory_dropdown],
            outputs=[chatbot, memory_state_display, context_display, status_display]
        )

    return dashboard



## Cell 10 — Launch the Dashboard
This will start the Gradio app and print step-by-step testing instructions in your Colab output.


In [11]:

# ================================
# CELL 10: Launch Dashboard
# ================================

print("Creating dashboard interface...")
dashboard = create_dashboard()

print("Launching Memory Testing Dashboard with Qwen...")
print("="*70)
print("TESTING INSTRUCTIONS:")
print("1. Select 'Episodic Memory' (most visual)")
print("2. Click '🚀 Start Chat'")
print("3. Ask: 'What is attention in transformers?'")
print("4. Ask: 'How does self-attention work?'")
print("5. Ask: 'What did we discuss about attention?'")
print("6. Watch the 'Memory State' and 'Retrieved Context' panels!")
print("7. Try different memory types and scenarios")
print("="*70)
print("🎉 Dashboard starting with Qwen LLM...")

# Launch the dashboard
dashboard.launch(
    share=True,
    debug=True,
    server_name="0.0.0.0",
    server_port=7860
)


Creating dashboard interface...
Launching Memory Testing Dashboard with Qwen...
TESTING INSTRUCTIONS:
1. Select 'Episodic Memory' (most visual)
2. Click '🚀 Start Chat'
3. Ask: 'What is attention in transformers?'
4. Ask: 'How does self-attention work?'
5. Ask: 'What did we discuss about attention?'
6. Watch the 'Memory State' and 'Retrieved Context' panels!
7. Try different memory types and scenarios
🎉 Dashboard starting with Qwen LLM...


/tmp/ipython-input-401190717.py:56: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://87d3823f37b698dd8e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> https://87d3823f37b698dd8e.gradio.live



---

## Teaching Notes & Troubleshooting

- **GPU strongly recommended**: Go to *Runtime → Change runtime type* and select a GPU.
- **Model download too slow?** Colab can occasionally be rate-limited. Rerun Cell 4–7.
- **FAISS errors on Mac M-chips** (local runs): In Colab this should be fine via `faiss-cpu`.
- **PDF extraction quirks**: Some PDFs return empty pages. We filter very short pages automatically.
- **No sources shown**: If retrieval returns nothing, check that embeddings were built in **Cell 7**.

### Pedagogical prompts
- Ask a series of related questions, then test **recall** (e.g., "What did we discuss earlier?").
- Switch topics, then return to the first topic; observe **context switching** behavior.
- Compare **Buffer vs Summary vs Episodic** with the same queries.

### Extending this notebook (optional)
- Swap in a larger Qwen or another HF-instruct model.
- Add multi-PDF ingestion for broader context.
- Experiment with temperature/top-p to see how generation style changes.

Happy experimenting! 🎉
